In [1]:
import pandas as pd
import json
import os

# Load the original Llama annotations
output_ann = pd.read_json(
    "output/annotations.jsonl",
    lines=True
)

# Load your manually reviewed JSON
human_ann = pd.read_json(
    "output/manual_review/manual_review_1015.json"
)

print("Original annotations:", len(output_ann))
print("Human-reviewed records:", len(human_ann))

print("\nOriginal annotation columns:")
print(output_ann.columns.tolist())

print("\nHuman annotation columns:")
print(human_ann.columns.tolist())

display(human_ann.head(3))

Original annotations: 63785
Human-reviewed records: 1015

Original annotation columns:
['response_id', 'question_id', 'fact_id', 'language', 'model_name', 'timestamp', 'judge_model', 'annotation_timestamp', 'correctness_label', 'hallucination_types', 'severity', 'claims', 'reason', 'confidence', 'review_required', 'judge_prompt_tokens', 'judge_output_tokens']

Human annotation columns:
['response_id', 'question_id', 'fact_id_response', 'language', 'model_name', 'domain_response', 'difficulty_response', 'question', 'ground_truth', 'response', 'source_url_response', 'source_url_reference', '_merge', 'human_correctness_label', 'human_hallucination_types', 'human_severity', 'human_reason', 'human_annotator', 'human_review_status']


,response_id,question_id,fact_id_response,language,model_name,domain_response,difficulty_response,question,ground_truth,response,source_url_response,source_url_reference,_merge,human_correctness_label,human_hallucination_types,human_severity,human_reason,human_annotator,human_review_status
0,c4428f654337cf4545d1,Q_HI_TEC_135,NaN,HI,GPT-5.6 Luna,NaN,NaN,एआरपीानेट के पहले नोड कब आपस में जुड़े थे और 1...,29 अक्टूबर 1969,29 अक्टूबर 1969 को और 1971 के अंत तक लगभग 15 स...,NaN,NaN,both,Correct,None,None,The response substantially preserves the refer...,Human1,Completed
1,351fe0060e7617f53616,Q_NE_GEO_098,NaN,NE,GPT-5.6 Luna,NaN,NaN,सहरी क्षेत्रका आधारमा चीनको राजधानी र सबैभन्दा...,बेइजिङ चीनको राजधानी हो; सहरी क्षेत्रका आधारमा...,राजधानी बेइजिङ हो र सहरी क्षेत्रका आधारमा सबैभ...,NaN,NaN,both,Partially Correct,Numerical Error; Factual Error,Moderate,At least one numeric value present in the grou...,Human1,Completed
2,4d7924ac2e1708d0200c,Q_NE_SCI_052,NaN,NE,GPT-5.6 Luna,NaN,NaN,विद्युत प्रवाह र चुम्बकीय क्षेत्रहरू कसरी सम्ब...,विद्युत प्रवाहले चुम्बकीय क्षेत्र उत्पन्न गर्छ...,विद्युत प्रवाहले चुम्बकीय क्षेत्र उत्पन्न गर्छ...,NaN,NaN,both,Correct,None,None,The response substantially preserves the refer...,Human1,Completed


In [7]:
# Keep all completed human reviews
human_completed = human_ann[
    human_ann["human_review_status"]
    .astype(str)
    .str.strip()
    .str.lower()
    .isin(["completed", "reviewed"])
].copy()

print("Human-reviewed records:", len(human_completed))

# Create DataFrame matching original annotation schema
human_for_merge = pd.DataFrame(
    index=range(len(human_completed)),
    columns=output_ann.columns
)

# Copy matching metadata
for col in output_ann.columns:
    if col in human_completed.columns:
        human_for_merge[col] = (
            human_completed[col].reset_index(drop=True)
        )

# Map human annotation fields
field_mapping = {
    "human_correctness_label": "correctness_label",
    "human_hallucination_types": "hallucination_types",
    "human_severity": "severity",
    "human_reason": "reason"
}

for human_col, original_col in field_mapping.items():
    human_for_merge[original_col] = (
        human_completed[human_col].reset_index(drop=True)
    )

# Preserve human annotation provenance
human_for_merge["annotation_source"] = "human"
human_for_merge["annotator_model"] = None
human_for_merge["human_annotator"] = (
    human_completed["human_annotator"].reset_index(drop=True)
)

# Set judge_model to identify human annotation
human_for_merge["judge_model"] = "Human"

print("Human annotations prepared:", len(human_for_merge))

display(
    human_for_merge[
        [
            "response_id",
            "question_id",
            "correctness_label",
            "hallucination_types",
            "severity",
            "reason",
            "annotation_source",
            "human_annotator"
        ]
    ].head(5)
)

Human-reviewed records: 1015
Human annotations prepared: 1015


,response_id,question_id,correctness_label,hallucination_types,severity,reason,annotation_source,human_annotator
0,c4428f654337cf4545d1,Q_HI_TEC_135,Correct,None,None,The response substantially preserves the refer...,human,Human1
1,351fe0060e7617f53616,Q_NE_GEO_098,Partially Correct,Numerical Error; Factual Error,Moderate,At least one numeric value present in the grou...,human,Human1
2,4d7924ac2e1708d0200c,Q_NE_SCI_052,Correct,None,None,The response substantially preserves the refer...,human,Human1
3,94a4b484ca5dd80ef0d1,Q_NE_HEA_174,Partially Correct,Omission,Minor,The response shares some reference content but...,human,Human1
4,faf064c25b6b4bf412de,Q_TA_CUL_127,Partially Correct,Omission,Minor,The response shares some reference content but...,human,Human1


In [8]:
original_for_merge = output_ann.copy()

original_for_merge["annotation_source"] = "llama_judge"
original_for_merge["annotator_model"] = "llama3.1:8b"
original_for_merge["human_annotator"] = None

print("Original annotations:", len(original_for_merge))
print("Human annotations:", len(human_for_merge))

Original annotations: 63785
Human annotations: 1015


In [9]:
final_annotations = pd.concat(
    [
        original_for_merge,
        human_for_merge
    ],
    ignore_index=True
)

print("Final annotation count:", len(final_annotations))
print("Unique response IDs:", final_annotations["response_id"].nunique())

display(final_annotations["annotation_source"].value_counts())

Final annotation count: 64800
Unique response IDs: 64800


C:\Users\CH.SC.U4CSE23160\AppData\Local\Temp\ipykernel_27336\1568928607.py:1: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  final_annotations = pd.concat(


annotation_source
llama_judge    63785
human           1015
Name: count, dtype: int64

In [10]:
assert len(final_annotations) == 64800

assert final_annotations["response_id"].nunique() == 64800

assert final_annotations["response_id"].isna().sum() == 0

assert (
    final_annotations["annotation_source"].value_counts().to_dict()
    == {
        "llama_judge": 63785,
        "human": 1015
    }
)

assert set(output_ann["response_id"]).isdisjoint(
    set(human_for_merge["response_id"])
) == True

print("SUCCESS: All 64,800 annotations are merged correctly.")

SUCCESS: All 64,800 annotations are merged correctly.


In [11]:
os.makedirs("output", exist_ok=True)

# Save final JSONL
final_annotations.to_json(
    "output/final_annotations.jsonl",
    orient="records",
    lines=True,
    force_ascii=False
)

# Save CSV
final_csv = final_annotations.copy()

for col in final_csv.columns:
    final_csv[col] = final_csv[col].apply(
        lambda x: json.dumps(x, ensure_ascii=False)
        if isinstance(x, (list, dict))
        else x
    )

final_csv.to_csv(
    "output/final_annotations.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Final JSONL and CSV saved successfully.")

Final JSONL and CSV saved successfully.
